# Changing Modules

## Setting up DSPy

### Load environment variables

In [1]:
import os

try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    from pathlib import Path

    if not Path('.env').exists():
        print("No .env file found. Please create one with OPENROUTER_API_KEY.")

    from dotenv import load_dotenv
    load_dotenv(override=True)

print(len(os.environ["OPENROUTER_API_KEY"]))

73


### Install Library

In [2]:
# ! uv add dspy
# ! pip install dspy

### Connecting to a language model

DSPy connects to language models with the `dspy.LM` class. To set up a language model, we provide a `"provider/model"` format string and an API key:


In [3]:
import dspy

# Pass the key explicitly...
lm = dspy.LM(
    "openrouter/openai/gpt-4.1-nano",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

dspy.configure(lm=lm)

## Change inference strategies by changing the module

In our previous examples, we used the `Predict` module to execute our signature.

![](../assets/changing_modules.png)

[Other modules](https://dspy.ai/diving-deeper/built-in-module-variants/) define different strategies for executing a task, and trying them out is very simple:

In [4]:
reasoning_haiku_bot = dspy.ChainOfThought("location, mood, season -> haiku")
result = reasoning_haiku_bot(location="Riyadh", mood="traffic", season="summer")
print(result.haiku)

Summer heatwaves stretch,  
Riyadh's streets breathe congestion,  
heat dulls the senses.


Instead of `Predict` we used `ChainOfThought`, a module that prompts the LM to reason before delivering a final answer. Our signature, the class-based `HaikuBot`, is the same. The function call, identical.

But behind the scenes, DSPy modified our signature to prompt the model to reason before producing its final poem. The newly added signature output field, `reasoning`, is now populated by the model’s rationale.

We can view this by printing `result.reasoning`:

In [5]:
print(result.reasoning)

Given the location is Riyadh during summer, the traffic is likely to be heavy and oppressive, with the heat intensifying the frustration. The season adds a sense of exhaustion yet resilience in enduring the crowded, hot roads. The haiku should evoke the image of bustling, congested streets under the scorching sun, capturing both the environment and mood.


`ChainOfThought` is simple, but nicely demonstrates the flexibility of DSPy’s module layer. [Other modules](https://dspy.ai/diving-deeper/built-in-module-variants/) implement other strategies, including model ensembles, coding sandboxes, and *tool-calling*, which the next section explores.


## 